# Bankruptcy label-time audit
## tl;dr
Local CSV has 78,682 firm-year rows, 8,971 firms, and 5,220 failed-labelled rows. All firms have constant status across their histories. A diagnostic candidate that counts only each failed firm's final observation yields 609 rows and matches the original paper's 20 annual event counts after shifting fyear by +1. This is supporting evidence, not independently verified filing dates or permission to overwrite labels.
Validation: Python code cells were executed sequentially on 2026-09-09. Jupyter-kernel execution and nbformat-library validation were not performed because nbformat, nbclient and ipykernel are absent in the project environment. JSON structure and code compilation were checked. Outputs are intentionally not embedded; run cells to reproduce.

## Context & Methods
Purpose: distinguish financial observation year, firm status, event year, and label availability. Read-only audit; no raw/results writes.
### Key Assumptions
The candidate last-observation rule is a diagnostic only. Absence of future records is not in general evidence of bankruptcy. No claim of exact filing dates or fiscal-report release dates is made.
Sources: original paper https://air.unipr.it/bitstream/11381/2933563/5/futureinternet-14-00244-v2.pdf, section 3 and Table 1 (p.5); upstream https://github.com/sowide/bankruptcy_dataset/tree/8bcd8db3b432b1fa6d65e26753b5c9fc6567438d. A separate in-memory download verified local CSV bytes identical to that commit: SHA256 d64ed85ae786d75113455ae238feb24015d19716c10ec1de3a90a52eddc7b04a. This offline notebook does not re-download upstream.

## Data
### 1. Locate and read the versioned raw data

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/bankruptcy/american_bankruptcy_dataset.csv').is_file()), None)
assert root is not None, 'Run from the repository or docs/notebooks directory'
raw_path = root / 'data/raw/bankruptcy/american_bankruptcy_dataset.csv'
raw_bytes = raw_path.read_bytes()
digest = hashlib.sha256(raw_bytes).hexdigest()
assert digest == 'd64ed85ae786d75113455ae238feb24015d19716c10ec1de3a90a52eddc7b04a', 'Data version changed; re-audit before comparing'
data = pd.read_csv(raw_path)
print('sha256:', digest)
print('shape:', data.shape)
print('columns:', data.columns.tolist())


## Results
### 2. Grain, labels, completeness and year validity

In [ ]:
raw_positive = data['status_label'].eq('failed')
profile = {
    'rows': len(data), 'firms': data['company_name'].nunique(),
    'missing_cells': int(data.isna().sum().sum()),
    'duplicate_firm_year': int(data.duplicated(['company_name', 'fyear']).sum()),
    'min_fyear': int(data.fyear.min()), 'max_fyear': int(data.fyear.max()),
    'integer_years': bool(data.fyear.mod(1).eq(0).all()),
    'raw_failed_rows': int(raw_positive.sum()),
    'firms_with_changing_status': int(data.groupby('company_name').status_label.nunique().gt(1).sum()),
}
print(profile)
print('firm status counts:', data.groupby('company_name').status_label.first().value_counts().to_dict())
assert set(data.status_label) == {'alive', 'failed'}
assert profile['duplicate_firm_year'] == 0
print(data.loc[data.company_name.eq('C_6'), ['company_name', 'fyear', 'status_label']].to_string(index=False))


### 3. Compare a diagnostic candidate with the original paper
Paper Table 1 lists outcome years 2000-2019 corresponding to financial feature years 1999-2018. Values below are a transcription of its event counts and are compared with all 20 local years. The candidate is not a production target.

In [ ]:
paper_event_counts = [3, 7, 10, 17, 29, 46, 40, 51, 59, 58, 23, 35, 25, 26, 28, 33, 33, 29, 21, 36]
last_observation = data.fyear.eq(data.groupby('company_name').fyear.transform('max'))
candidate = raw_positive & last_observation
comparison = data.assign(raw_positive=raw_positive, candidate=candidate).groupby('fyear').agg(
    n_rows=('company_name', 'size'), raw_failed_rows=('raw_positive', 'sum'), candidate_last_failed=('candidate', 'sum'))
comparison['outcome_year_under_paper_convention'] = comparison.index.astype(int) + 1
comparison['paper_event_count'] = paper_event_counts
comparison['matches_paper'] = comparison.candidate_last_failed.eq(comparison.paper_event_count)
print(comparison.to_string())
print('all_20_years_match:', bool(comparison.matches_paper.all()))
print('raw_positive_rate:', float(raw_positive.mean()))
print('candidate_positive_rate:', float(candidate.mean()))
print('additional_raw_positive_rows:', int(raw_positive.sum() - candidate.sum()))
assert comparison.matches_paper.all()
assert candidate.sum() == 609
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == digest, 'Raw data changed during audit'


## Takeaways
The file has valid annual observation keys but no explicit event/filing/label-availability date. Literal failed-status classification and next-year event prediction are different targets: 5,220 vs candidate 609 positives (6.6343% vs 0.7740%). The full annual agreement supports a candidate reconstruction for further verification, not an independently validated prospective label contract. Preserve raw; confirm event mapping, reporting delay and negative follow-up/censoring before creating a separate derived target and retraining. See ../Study3_B路線_重疊窗口持續集成提案.md for the revised B protocol.